In [1]:
import numpy as np
import healpy as hp
import matplotlib.pylab as plt
%matplotlib inline
import rubin_sim.splat as splat
import copy 

from rubin_scheduler.utils import mjd2dayobs, hpid2_ra_dec

import sqlite3
import pandas as pd
from os.path import basename

from desi_utils import get_desi_stripes, is_in_ra_range

In [2]:
# Define a report element that we want to run. Needs branch maf-proto of rubin_sim to run
class DesiComleteElement(splat.BaseReportElement):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.desi_stripes = get_desi_stripes()
    
    def compute_elements(self,visits_df=None, visits_array=None, nside=64, rubin_to_desi_yr_offset=1, bands='ugr'):
        
        info = {"data_source": self.run_name}
        info["observations_subset"] = "\nNo ToO, No DDf, expt > 20"
        
        is_too = np.array(["ToO" in note for note in visits_array["scheduler_note"]])
        is_ddf = np.array(["DD" in note for note in visits_array["scheduler_note"]])
        stripes = self.desi_stripes
        df = pd.DataFrame
        rows = []
        ra,dec = hpid2_ra_dec(nside, np.arange(hp.nside2npix(nside)))
        for band in bands:
            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20) & (visits_array["band"] == band))
            sub_data = visits_array[indx]
            sl = splat.Slicer(nside=nside, missing=0)
            times = np.arange(3653)
            metric= splat.AccumulateCountMetric(times)
            hp_array, info = sl(sub_data, metric, info=info)

            for ramin, ramax, decmin, decmax, nu, year,name in zip(stripes["ramins"], stripes["ramaxs"],
                                     stripes["decmins"], stripes["decmaxs"],
                                     stripes["nus"], stripes["yrs"], stripes["names"]):  
                indx = np.where((dec > decmin) & (dec < decmax) & (is_in_ra_range(ra, ramin, ramax)))[0]
                night = int(np.round((year+rubin_to_desi_yr_offset)*365.25))
                median_counts = np.median(hp_array[indx, night])
                night_diff = np.abs(visits_array["night"] - night)
                time_indx = np.where(night_diff == night_diff.min())[0].min()
                
                mjd = visits_array["observationStartMJD"][time_indx]
                if band == "u":
                    rows.append({"name": name, "band": band, "median_visits": median_counts,
                                 "desired_visits": nu, "dayobs":mjd2dayobs(mjd)})
                else:
                    rows.append({"name": name, "band": band, "median_visits": median_counts,
                                 "desired_visits": "?", "dayobs":mjd2dayobs(mjd)})
                
        df = pd.DataFrame(rows)
        return None, None, df


In [3]:
db_files = ['baseline_v5.3.7_10yrs.db', 'three_stripes_v5.3.7_10yrs.db',
            'three_stripes_kr_v5.3.7_10yrs.db',
            'desi_tskr_v5.3.7_10yrs.db',
            'desi_stripes_delayed_v5.3.7_10yrs.db',
            'desi_stripes_v5.3.7_10yrs.db',
            ]

In [4]:
tables = []
for filename in db_files:
    element = DesiComleteElement(observations=filename)
    hp_arrays, infos, summary = element()
    tables.append(summary)

/Users/yoachim/anaconda3/envs/rubin_14/lib/python3.14/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 1 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [5]:
i = 3
print(db_files[i])
tables[i]

desi_tskr_v5.3.7_10yrs.db


,name,band,median_visits,desired_visits,dayobs
0,NGC-1,u,43.0,37.3,20310615
1,NGC-2,u,35.0,32.4,20300610
2,NGC-3,u,32.0,29.8,20290612
3,NGC-4,u,31.0,29.1,20290612
4,NGC-5,u,31.0,29.3,20310615
5,NGC-6,u,29.0,24.6,20300610
6,NGC-7,u,25.0,23.1,20280614
7,NGC-8,u,23.0,20.4,20280614
8,NGC-9,u,32.0,19.3,20320612
9,SGC-1,u,33.0,33.2,20310615
